# Data Preprocessing for Sentiment Analysis
##### Ömer Faruk Merey - Middle East Technical University

This notebook prepares the customer service conversation dataset for training. Tokenization is handled separately in each training notebook since DeBERTa and Longformer require different strategies.

**Steps:**
1. Load & inspect data
2. Feature selection
3. Text cleaning
4. Label encoding
5. Train/validation split (no test leakage)
6. Token length analysis
7. Compute class weights
8. Save preprocessed data

In [1]:
import pandas as pd
import numpy as np
import re
import os

train_df = pd.read_csv('../dataset/train.csv')
test_df = pd.read_csv('../dataset/test.csv')

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
train_df.head(3)

Train: (970, 11), Test: (30, 11)


,issue_area,issue_category,issue_sub_category,issue_category_sub_category,customer_sentiment,product_category,product_sub_category,issue_complexity,agent_experience_level,agent_experience_level_desc,conversation
0,Login and Account,Mobile Number and Email Verification,Verification requirement for mobile number or ...,Mobile Number and Email Verification -> Verifi...,neutral,Appliances,Oven Toaster Grills (OTG),medium,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox Customer...
1,Cancellations and returns,Pickup and Shipping,Reasons for being asked to ship the item,Pickup and Shipping -> Reasons for being asked...,neutral,Electronics,Computer Monitor,less,junior,"handles customer inquiries independently, poss...",Agent: Thank you for calling BrownBox customer...
2,Cancellations and returns,Replacement and Return Process,Inability to click the 'Cancel' button,Replacement and Return Process -> Inability to...,neutral,Appliances,Juicer/Mixer/Grinder,medium,experienced,"confidently handles complex customer issues, e...",Agent: Thank you for calling BrownBox Customer...


## 1. Feature Selection

From the EDA, we observed:
- `conversation` is the primary text feature for the transformer
- `issue_category` has the strongest correlation with sentiment (Cramér's V = 0.739)
- `issue_area` also correlates well (V = 0.295)
- `issue_category_sub_category` is redundant (combines issue_category + issue_sub_category)
- `agent_experience_level_desc` is redundant (describes agent_experience_level)

**Decision:** Use `conversation` as the main input. Drop redundant columns.

In [2]:
# Drop redundant features
drop_cols = ['issue_category_sub_category', 'agent_experience_level_desc']

train_df = train_df.drop(columns=drop_cols)
test_df = test_df.drop(columns=drop_cols)

print(f"Remaining columns: {list(train_df.columns)}")

Remaining columns: ['issue_area', 'issue_category', 'issue_sub_category', 'customer_sentiment', 'product_category', 'product_sub_category', 'issue_complexity', 'agent_experience_level', 'conversation']


## 2. Text Cleaning

The `conversation` column contains agent-customer dialogues. We apply minimal cleaning to preserve semantic meaning for the transformer:
- Normalize whitespace (collapse multiple spaces/newlines)
- Keep speaker labels (`Agent:`, `Customer:`) as they provide useful context
- Remove any leading/trailing whitespace

**Justification:** Transformer tokenizers (e.g., BERT) handle punctuation, casing, and special characters well, so aggressive cleaning (lowercasing, removing punctuation) would remove useful signal.

In [3]:
def clean_conversation(text):
    """Minimal text cleaning for transformer input."""
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['conversation'] = train_df['conversation'].apply(clean_conversation)
test_df['conversation'] = test_df['conversation'].apply(clean_conversation)

# Verify
print(f"Sample cleaned text (first 200 chars):")
print(train_df['conversation'].iloc[0][:200])

Sample cleaned text (first 200 chars):
Agent: Thank you for calling BrownBox Customer Support. My name is Tom. How may I assist you today? Customer: Hi Tom, I'm trying to log in to my account to purchase an Oven Toaster Grill (OTG), but I'


## 3. Label Encoding

Encode `customer_sentiment` to numeric labels for model training.

In [4]:
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
label_map_inv = {v: k for k, v in label_map.items()}

train_df['label'] = train_df['customer_sentiment'].map(label_map)
test_df['label'] = test_df['customer_sentiment'].map(label_map)

print("Label mapping:", label_map)
print(f"\nTrain label distribution:\n{train_df['label'].value_counts().sort_index().rename(label_map_inv)}")
print(f"\nTest label distribution:\n{test_df['label'].value_counts().sort_index().rename(label_map_inv)}")

Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}

Train label distribution:
label
negative    411
neutral     542
positive     17
Name: count, dtype: int64

Test label distribution:
label
negative    10
neutral     10
positive    10
Name: count, dtype: int64


## 4. Train / Validation Split

Split `train.csv` into train and validation sets using stratified sampling to preserve class distribution.

**No test data is used here — test.csv is only for final evaluation.**

In [5]:
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)

print(f"Train split: {train_split.shape[0]} samples")
print(f"Val split:   {val_split.shape[0]} samples")
print(f"Test set:    {test_df.shape[0]} samples")

# Verify stratification
splits_dist = pd.DataFrame({
    'Train %': train_split['label'].value_counts(normalize=True).sort_index() * 100,
    'Val %': val_split['label'].value_counts(normalize=True).sort_index() * 100,
    'Test %': test_df['label'].value_counts(normalize=True).sort_index() * 100
}).round(1)
splits_dist.index = splits_dist.index.map(label_map_inv)
splits_dist

Train split: 776 samples
Val split:   194 samples
Test set:    30 samples


,Train %,Val %,Test %
label,,,
negative,42.4,42.3,33.3
neutral,55.9,55.7,33.3
positive,1.7,2.1,33.3


## 5. Token Length Analysis

Analyze token counts to inform max_length choices for each model:
- **DeBERTa** (max 512): will use head+tail truncation strategy
- **Longformer** (max 4096): can fit full conversations

In [ ]:
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
token_lengths = train_df['conversation'].apply(lambda x: len(tokenizer.encode(x, add_special_tokens=True)))

print("Token length stats:")
print(f"  Mean:   {token_lengths.mean():.0f}")
print(f"  Median: {token_lengths.median():.0f}")
print(f"  95th %: {token_lengths.quantile(0.95):.0f}")
print(f"  99th %: {token_lengths.quantile(0.99):.0f}")
print(f"  Max:    {token_lengths.max():.0f}")

pct_over_512 = (token_lengths > 512).mean() * 100
print(f"\n  Conversations exceeding 512 tokens: {pct_over_512:.1f}%")
print(f"  Conversations exceeding 1024 tokens: {(token_lengths > 1024).mean() * 100:.1f}%")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(token_lengths, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(512, color='r', linestyle='--', label='DeBERTa max (512)')
ax.axvline(4096, color='g', linestyle='--', label='Longformer max (4096)')
ax.axvline(token_lengths.quantile(0.95), color='orange', linestyle='--',
           label=f'95th pct ({token_lengths.quantile(0.95):.0f})')
ax.set_xlabel('Token count')
ax.set_ylabel('Frequency')
ax.set_title('Token Length Distribution (train set)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Compute Class Weights

The dataset is heavily imbalanced (positive class has only 17 samples / 1.7%). We compute class weights to use in weighted cross-entropy loss during training.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_split['label'].values
)

class_weight_dict = {label_map_inv[i]: round(w, 3) for i, w in enumerate(class_weights)}
print("Class weights (for weighted cross-entropy):")
for cls, w in class_weight_dict.items():
    print(f"  {cls}: {w}")

print(f"\nThese weights will be passed to CrossEntropyLoss in both training notebooks.")

## 7. Save Preprocessed Data

Save cleaned CSVs and class weights. Tokenization is deferred to each training notebook.

In [ ]:
import json

output_dir = '../dataset/processed'
os.makedirs(output_dir, exist_ok=True)

# Save cleaned CSVs

train_split.to_csv(f'{output_dir}/train_split.csv', index=False)
val_split.to_csv(f'{output_dir}/val_split.csv', index=False)
test_df.to_csv(f'{output_dir}/test_processed.csv', index=False)

# Save metadata (label map, class weights)
metadata = {
    'label_map': label_map,
    'class_weights': class_weights.tolist(),
    'train_size': len(train_split),
    'val_size': len(val_split),
    'test_size': len(test_df)
}
with open(f'{output_dir}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to {output_dir}/")
for fname in sorted(os.listdir(output_dir)):
    size = os.path.getsize(f'{output_dir}/{fname}') / 1024
    print(f"  {fname}: {size:.1f} KB")

## Summary

| Step | Action | Justification |
|------|--------|---------------|
| Feature selection | Dropped `issue_category_sub_category`, `agent_experience_level_desc` | Redundant (derived from other columns) |
| Text cleaning | Normalized whitespace only | Transformers handle punctuation/casing natively |
| Label encoding | negative=0, neutral=1, positive=2 | Standard ordinal mapping |
| Train/Val split | 80/20 stratified split from train.csv | Preserves class distribution, no test leakage |
| Class weights | Computed balanced weights per class | Positive class has only 1.7% of samples |
| Output | Cleaned CSVs + metadata JSON | Tokenization deferred to training notebooks |

**Tokenization strategy per model:**

| Model | Max Tokens | Strategy |
|-------|-----------|----------|
| DeBERTa-v3-base | 512 | Head+tail truncation (first 256 + last 256 tokens) to capture both problem statement and resolution |
| Longformer-base | 4096 | Full conversation fits without truncation |